In [3]:
import pandas as pd
from IPython.display import display
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', True)

In [13]:
y = pd.read_excel("final_y.xlsx")

In [5]:
df = pd.read_excel("final_df.xlsx")

In [7]:
df.shape

(2216, 50)

In [9]:
df.head()

,Gross Profit Margin - %,Net Income after Minority Interest,Total Capital,Free Cash Flow,Revenue from Business Activities - Total,Revenue from Business Activities - Total.1,Revenue from Business Activities - Total.2,Revenue from Business Activities - Total.3,Current Ratio,Price To Book Value Per Share (Daily Time Series Ratio),...,word2vec_61,word2vec_71,word2vec_78,word2vec_81,word2vec_86,word2vec_88,word2vec_92,word2vec_94,word2vec_99,Avg_Sentiment_Label
0,-0.011713,-0.028612,-0.038890,-0.016175,-0.046135,-0.044215,-0.044582,-0.047133,-0.399799,0.275817,...,0.235085,-0.061244,0.158181,0.102279,-0.678453,-0.300352,0.015876,-0.876598,-0.242376,-0.001212
1,-0.313778,-0.028733,-0.039166,-0.016180,-0.046332,-0.044386,-0.044758,-0.047344,0.212090,0.022465,...,-0.485344,0.916858,0.935876,-0.855071,0.330537,-0.717210,-0.317035,1.171323,-0.760501,0.891525
2,0.480681,-0.028944,-0.038826,-0.014838,-0.045340,-0.043544,-0.043923,-0.046360,-0.401813,0.020841,...,0.656171,0.531424,0.115371,-0.477624,-1.325744,-0.535916,0.343792,-0.456287,-0.352434,0.810685
3,-1.146303,-0.029045,-0.038907,-0.016298,-0.046203,-0.044137,-0.044470,-0.046961,0.432580,0.022458,...,1.187239,1.033560,1.208084,-1.226435,-1.954682,-1.301515,0.199861,0.056993,-1.349756,0.474690
4,0.918035,-0.028783,-0.038870,-0.016336,-0.046287,-0.044351,-0.044726,-0.047311,1.807264,0.022920,...,0.440908,0.462585,0.527929,-0.453128,-0.685875,-0.107520,-0.650180,-0.522044,-0.619982,1.447306


In [15]:
y.head()

,Label
0,1
1,1
2,1
3,1
4,1


In [17]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Features and target
X = df
y = y

# Split first
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {X_train.shape}, After SMOTE: {X_train_sm.shape}")
print(f"Class balance after SMOTE:\n{y_train_sm.value_counts()}")


Before SMOTE: (1772, 50), After SMOTE: (3252, 50)
Class balance after SMOTE:
Label
0        1626
1        1626
Name: count, dtype: int64


In [19]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Split full dataset FIRST (before removing any columns)
X = df.copy()
y = y.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Apply SMOTE ONLY on training set
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# 3. Identify which are word2vec and sentiment columns
word2vec_cols = [col for col in X.columns if 'word2vec' in col]
sentiment_cols = ['Avg_Sentiment_Label']

# 4. Create two datasets
# (a) Full (all features)
X_train_full = X_train_sm.copy()
X_test_full = X_test.copy()

# (b) Only numerical financial features (drop word2vec + sentiment)
X_train_num = X_train_sm.drop(columns=word2vec_cols + sentiment_cols)
X_test_num = X_test.drop(columns=word2vec_cols + sentiment_cols)

# 5. Fit XGBoost on FULL dataset
xgb_full = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    scale_pos_weight=1,  # already balanced after SMOTE
    use_label_encoder=False,
    eval_metric="logloss"
)
xgb_full.fit(X_train_full, y_train_sm)

# 6. Fit XGBoost on NUMERICAL dataset
xgb_num = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    scale_pos_weight=1,
    use_label_encoder=False,
    eval_metric="logloss"
)
xgb_num.fit(X_train_num, y_train_sm)

# 7. Predictions
y_pred_full = xgb_full.predict(X_test_full)
y_pred_num = xgb_num.predict(X_test_num)

# 8. Evaluation
print("===== Full Dataset (Numerical + Word2Vec + Sentiment) =====")
print(classification_report(y_test, y_pred_full))
print(confusion_matrix(y_test, y_pred_full))
print("Accuracy:", accuracy_score(y_test, y_pred_full))

print("\n===== Only Numerical Financial Features =====")
print(classification_report(y_test, y_pred_num))
print(confusion_matrix(y_test, y_pred_num))
print("Accuracy:", accuracy_score(y_test, y_pred_num))


/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [21:34:58] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


===== Full Dataset (Numerical + Word2Vec + Sentiment) =====
              precision    recall  f1-score   support

           0       0.95      0.92      0.94       407
           1       0.35      0.46      0.40        37

    accuracy                           0.89       444
   macro avg       0.65      0.69      0.67       444
weighted avg       0.90      0.89      0.89       444

[[376  31]
 [ 20  17]]
Accuracy: 0.8851351351351351

===== Only Numerical Financial Features =====
              precision    recall  f1-score   support

           0       0.93      0.78      0.84       407
           1       0.12      0.32      0.17        37

    accuracy                           0.74       444
   macro avg       0.52      0.55      0.51       444
weighted avg       0.86      0.74      0.79       444

[[316  91]
 [ 25  12]]
Accuracy: 0.7387387387387387


/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [21:34:59] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [23]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
from scipy.stats import ttest_rel
import numpy as np

# Assume df is your final dataset
X = df
y = y  # your label column

# Split into two versions
numerical_features = [col for col in df.columns if not col.startswith('word2vec') and 'Sentiment' not in col]
X_full = X  # Full: all features
X_numerical = X[numerical_features]  # Only numerical features

# Initialize cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# To store results
f1_full_model = []
f1_numerical_model = []

for train_idx, test_idx in skf.split(X_full, y):
    # Train/Test split
    X_train_full, X_test_full = X_full.iloc[train_idx], X_full.iloc[test_idx]
    X_train_num, X_test_num = X_numerical.iloc[train_idx], X_numerical.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # SMOTE only on training set
    smote = SMOTE(random_state=42)
    X_train_full_sm, y_train_sm = smote.fit_resample(X_train_full, y_train)
    X_train_num_sm, y_train_num_sm = smote.fit_resample(X_train_num, y_train)

    # Train full model
    model_full = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
    model_full.fit(X_train_full_sm, y_train_sm)
    y_pred_full = model_full.predict(X_test_full)
    f1_full_model.append(f1_score(y_test, y_pred_full))
    
    # Train numerical-only model
    model_num = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
    model_num.fit(X_train_num_sm, y_train_num_sm)
    y_pred_num = model_num.predict(X_test_num)
    f1_numerical_model.append(f1_score(y_test, y_pred_num))

# Paired t-test
stat, p_value = ttest_rel(f1_full_model, f1_numerical_model)
print(f"Paired t-test: t-statistic = {stat:.4f}, p-value = {p_value:.4f}")

print("\nAverage F1 Full Model:", np.mean(f1_full_model))
print("Average F1 Numerical Only:", np.mean(f1_numerical_model))


/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [21:47:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [21:47:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [21:47:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [21:47:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packa

Paired t-test: t-statistic = 14.1855, p-value = 0.0001

Average F1 Full Model: 0.3383801088873552
Average F1 Numerical Only: 0.19375431595542988


/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [21:47:23] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [25]:
# Let's retrain the final model on full dataset to see wrong predictions

# Final split
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, stratify=y, random_state=42
)

# Apply SMOTE to training
smote = SMOTE(random_state=42)
X_train_full_sm, y_train_sm = smote.fit_resample(X_train_full, y_train)

# Train model
model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
model.fit(X_train_full_sm, y_train_sm)

# Predict
y_pred = model.predict(X_test_full)

# Error Analysis
errors = X_test_full.copy()
errors['True_Label'] = y_test
errors['Predicted_Label'] = y_pred
errors['Correct'] = (errors['True_Label'] == errors['Predicted_Label'])

# Analyze
print("Number of mistakes:", (~errors['Correct']).sum())

# Look at some wrong examples
wrong_predictions = errors[errors['Correct'] == False]
print(wrong_predictions[['True_Label', 'Predicted_Label']].head())


Number of mistakes: 46
      True_Label  Predicted_Label
129            1                0
2057           0                1
155            1                0
1401           0                1
575            0                1


/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [21:48:36] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
